# 튜닝: H3 λ·Ridge λ·트리 수 고정 vs 해마다 선택

조정값을 지금처럼 **해마다 직전 1년(내부 검증)으로 고르는 것**과 **16년 내내 하나로 고정하는 것** 중 어느 쪽이 나은지 비교한다.
고정 1등이 해마다 선택보다 192개월 평균 IC로 **0.002 넘게** 좋을 때만 고정으로 바꾼다.

| | 선형 | 부스팅 |
| --- | --- | --- |
| 모델 | Ridge, 목표값 월별 z점수 | LightGBM 순위 회귀(백분위·MSE), 깊이 3, 학습률 0.01 |
| 입력 | [Size, B/M, Mom, σ], 매달 z점수 | [Size, B/M, Mom, σ], 매달 순위 [−1, 1] |
| 손실 | Huber①(δ = 99.9% 분위수), Huber②(δ = 1.35σ) | MSE |
| 고정 후보 | H3 10개 × Ridge λ 8개(0.0001~10 + 100·1000) = 80, 손실 2개 → 160 | H3 3개(span 12·36·60) × 트리 10·20·…·1000(100개) = 300 |
| 해마다 선택 | 손실마다 λ 후보 6개(지금)·8개 | 지금 방식(트리 1~1000 × H3 3개) |
| H3만 고정 | — | H3 1·3·5년 각각 고정, 트리 수만 해마다 선택(3개) → H3끼리 비교, 개선이 H3 덕인지 트리 수 덕인지 |

- Fold 16개(바깥 검증 2006~2021), 학습 2003년부터 확장창, 내부 검증 직전 1년, 고른 뒤 재학습 없음. 채점 133,876행 192개월.
- 같은 결과로 많은 고정 후보 중 1등을 고르므로 1등 값은 운으로 조금 부풀려진다 → 0.002 기준과 1등 주변이 평평한지를 함께 본다.
- 공용 코드는 고치지 않는다. 로컬 실행 약 15분(입력 준비 4분, 선형 2분, 부스팅 7~12분).
- 결과: `outputs/model_results/<AUTHOR>/tuning_fixed_linear.csv`, `tuning_fixed_lgb.csv`.

In [ ]:
import os, sys, time
from pathlib import Path

AUTHOR = ''        # 결과 csv 위치 outputs/model_results/<AUTHOR>/ (영문)
N_THREADS = 6      # LightGBM CPU 스레드

if not AUTHOR:
    raise ValueError('AUTHOR를 채운다.')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from src.evaluation import model_test as mt
from src.models import boosting, linear

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
FOLDS = mt.plan_folds(config['MODEL_GRID'])
OUT_DIR = ROOT / 'outputs' / 'model_results' / AUTHOR
OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 200, 'display.max_rows', 60, 'display.float_format', '{:.4f}'.format)


def score(val, pred):
    """192개월 IC: 평균, ICIR, IC>0 비율. 한 달 예측이 모두 같으면 그 달 IC 0(우리 결과와 같은 계산)."""
    t = mt.monthly_ic(val.target_month.to_numpy(), np.asarray(pred, float), val.y.to_numpy(float))
    ic = t.ic.where(t.reason.ne('constant_score'), 0.0)
    return {'months': int(ic.notna().sum()), 'mean_ic': ic.mean(), 'icir': ic.mean() / ic.std(), 'ic_pos': (ic > 0).mean()}


def persistence(values) -> float:
    """해마다 고른 값이 바로 전 해와 같은 비율."""
    v = list(values)
    return float(np.mean([a == b for a, b in zip(v[1:], v[:-1])]))


print('ROOT:', ROOT, '| 결과:', OUT_DIR, '| Fold', len(FOLDS))

## 1. 선형 (약 4분)

Fold마다 H3 10개 × 손실 2개 × Ridge λ 8개를 학습하고(λ는 작은 것부터, 앞 λ의 해를 다음 시작점으로 — 공용 코드와 같은 순서),
내부 검증 IC·학습 IC·바깥 검증 예측을 모두 남긴다. 해마다 선택 = 내부 검증 IC 최고, 같으면 λ 큰 쪽 → H3 긴 기간(공용 규칙).
λ 후보 6개로 해마다 고른 결과는 기존 공용 실행(Huber① 0.0903)과 같아야 한다.

In [ ]:
t0 = time.perf_counter()
prep_z = mt.prepare_inputs(ROOT, config, input_transform='zscore')
data_z, labels = prep_z['data'], prep_z['sigma_labels']
decay_of = mt._decay_of(prep_z)
val_z = mt.validation_rows(prep_z, config)
print(f'입력 준비(z점수) {time.perf_counter() - t0:.0f}초, H3 후보 {labels}')

LAMBDAS = (0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)
LOSSES = ('huber_q999', 'huber_sigma135')
TARGET = 'monthly_zscore'
preds_lin, inner_lin = {}, []
t0 = time.perf_counter()
for f in FOLDS:
    m = mt.fold_rows(prep_z, f)
    ctx = mt.FoldContext(data_z, m['fit'], m['inner'], m['out'])
    out_mask = val_z.fold.eq(f['fold']).to_numpy()
    y_fit = ctx.y(TARGET)
    for loss in LOSSES:
        for lab in labels:
            cols = ['x_size', 'x_bm', 'x_mom', 'x_sigma_' + lab]
            X = ctx.X(cols, 'fit')
            delta = ctx.q999(TARGET) if loss == 'huber_q999' else ctx.sigma2(cols, TARGET)['delta']
            warm = None
            for lam in LAMBDAS:
                fit = linear.fit_linear(X, y_fit, model='ridge', loss=loss, ridge_lambda=lam, delta=delta, start=warm)
                warm = (fit['intercept'], fit['coef'])
                inner_lin.append({'fold': f['fold'], 'loss': loss, 'h3_label': lab, 'decay': decay_of[lab], 'ridge_lambda': lam,
                                  'inner_ic': ctx.scorer.score(linear.predict_linear(fit, ctx.X(cols, 'inner')))[0],
                                  'train_ic': ctx.train_scorer.score(linear.predict_linear(fit, X))[0]})
                key = (loss, lab, lam)
                preds_lin.setdefault(key, np.full(len(val_z), np.nan))[out_mask] = linear.predict_linear(fit, ctx.X(cols, 'out'))
    print(f"  Fold {f['fold']:>2} ({f['valid_year']}) {time.perf_counter() - t0:.0f}초")
inner_lin = pd.DataFrame(inner_lin)

In [ ]:
rows = []
for (loss, lab, lam), p in preds_lin.items():
    tr = inner_lin[(inner_lin.loss == loss) & (inner_lin.h3_label == lab) & (inner_lin.ridge_lambda == lam)].train_ic.mean()
    rows.append({'kind': 'fixed', 'loss': loss, 'h3_label': lab, 'ridge_lambda': lam, **score(val_z, p), 'train_ic': tr})
for loss in LOSSES:
    for name, allowed in (('yearly_6lambdas', LAMBDAS[:6]), ('yearly_8lambdas', LAMBDAS)):
        cand = inner_lin[(inner_lin.loss == loss) & inner_lin.ridge_lambda.isin(allowed)]
        pick = (cand.sort_values(['fold', 'inner_ic', 'ridge_lambda', 'decay'], ascending=[True, False, False, False])
                .groupby('fold').head(1).sort_values('fold'))
        p = np.full(len(val_z), np.nan)
        for r in pick.itertuples():
            mask = val_z.fold.eq(r.fold).to_numpy()
            p[mask] = preds_lin[(loss, r.h3_label, r.ridge_lambda)][mask]
        rows.append({'kind': name, 'loss': loss, 'h3_label': '|'.join(pick.h3_label),
                     'ridge_lambda': '|'.join(f'{x:g}' for x in pick.ridge_lambda), **score(val_z, p), 'train_ic': pick.train_ic.mean(),
                     'persist_h3': persistence(pick.h3_label), 'random_h3': 1 / len(labels),
                     'persist_lambda': persistence(pick.ridge_lambda), 'random_lambda': 1 / len(allowed)})
lin = pd.DataFrame(rows)
lin.to_csv(OUT_DIR / 'tuning_fixed_linear.csv', index=False, encoding='utf-8-sig')
print('저장:', OUT_DIR / 'tuning_fixed_linear.csv')
for loss in LOSSES:
    part = lin[lin.loss.eq(loss)]
    fixed = part[part.kind.eq('fixed')]
    print(f'[{loss}] 해마다 선택')
    display(part[part.kind.ne('fixed')][['kind', 'mean_ic', 'icir', 'ic_pos', 'train_ic', 'persist_h3', 'random_h3',
                                         'persist_lambda', 'random_lambda']])
    print(f'[{loss}] 고정 평균 IC (행 H3, 열 Ridge λ)')
    display(fixed.pivot(index='h3_label', columns='ridge_lambda', values='mean_ic').reindex(labels))

## 2. 부스팅 (약 10분)

Fold마다 H3 3개로 1000트리를 한 번씩 학습(48회). 바깥 검증 예측은 10트리 묶음을 누적해 10·20·…·1000트리 100가지를 한 번에 만든다
(LightGBM은 묶음 합 = 전체 예측). 해마다 선택 = 공용 코드 `choose_boosting`(트리 1~1000 × H3 3개 중 내부 검증 IC 최고,
같으면 트리 적은 쪽 → H3 긴 기간). 해마다 선택 결과는 기존 공용 실행(0.0863)과 같아야 한다.
H3만 고정 = 그 H3의 학습 하나에서 트리 수만 해마다 고른다(같은 규칙: 내부 검증 IC 최고, 같으면 트리 적은 쪽).

In [ ]:
t0 = time.perf_counter()
prep_r = mt.prepare_inputs(ROOT, config)                 # 부스팅은 순위 입력
data_r = prep_r['data']
labels_b = mt.boosting_labels(prep_r)
decay_b = mt._decay_of(prep_r)
val_r = mt.validation_rows(prep_r, config)
print(f'입력 준비(순위) {time.perf_counter() - t0:.0f}초, H3 후보 {labels_b}')

TREES = list(range(10, 1001, 10))
SETTING = dict(target='percentile', loss='mse', depth=3, learning_rate=0.01)
preds_b = {(lab, k): np.full(len(val_r), np.nan) for lab in labels_b for k in TREES}
train_b, picks_b = [], []
pred_yearly = np.full(len(val_r), np.nan)
pred_h3fixed = {lab: np.full(len(val_r), np.nan) for lab in labels_b}      # H3 고정, 트리 수만 해마다
picks_h3fixed = []
months_r = data_r.target_month.to_numpy()
t0 = time.perf_counter()
for f in FOLDS:
    m = mt.fold_rows(prep_r, f)
    ctx = mt.FoldContext(data_r, m['fit'], m['inner'], m['out'])
    out_mask = val_r.fold.eq(f['fold']).to_numpy()
    fits = []
    for lab in labels_b:
        cols = ['x_size', 'x_bm', 'x_mom', 'x_sigma_' + lab]
        fit = boosting.fit_boosting('lgb_rankreg', ctx.X(cols, 'fit'), ctx.y('percentile'), ctx.m_fit, **SETTING,
                                    inner_X=ctx.X(cols, 'inner'), inner_months=months_r[m['inner']],
                                    inner_score=lambda p: ctx.scorer.score(p)[0], train_score=lambda p: ctx.train_scorer.score(p)[0],
                                    n_threads=N_THREADS)
        Xo = ctx.X(cols, 'out')
        cum = np.zeros(len(Xo))
        for k in TREES:                                   # 10트리 묶음 누적 = k트리 예측
            cum += fit['booster'].predict(Xo, start_iteration=k - 10, num_iteration=10, raw_score=True)
            preds_b[(lab, k)][out_mask] = cum
        train_b += [{'fold': f['fold'], 'h3_label': lab, 'trees': k, 'train_ic': fit['train_curve'][k // 10 - 1]} for k in TREES]
        fits.append({'h3_label': lab, 'decay': decay_b[lab], 'fitted': fit, 'curve': fit['inner_curve'], 'columns': cols})
    for x in fits:                                            # H3만 고정: 그 H3 학습에서 트리 수만 고름
        _, k, _ = mt.choose_boosting([x])
        pred_h3fixed[x['h3_label']][out_mask] = boosting.predict_boosting(x['fitted'], ctx.X(x['columns'], 'out'), k)
        picks_h3fixed.append({'fold': f['fold'], 'h3_label': x['h3_label'], 'trees': k,
                              'train_ic': x['fitted']['train_curve'][-(-k // 10) - 1]})
    chosen, trees, ties = mt.choose_boosting(fits)
    pred_yearly[out_mask] = boosting.predict_boosting(chosen['fitted'], ctx.X(chosen['columns'], 'out'), trees)
    picks_b.append({'fold': f['fold'], 'h3_label': chosen['h3_label'], 'trees': trees,
                    'train_ic': chosen['fitted']['train_curve'][-(-trees // 10) - 1]})
    print(f"  Fold {f['fold']:>2} ({f['valid_year']}) 고른 값 {chosen['h3_label']}·트리 {trees}, {time.perf_counter() - t0:.0f}초")
train_b, picks_b, picks_h3fixed = pd.DataFrame(train_b), pd.DataFrame(picks_b), pd.DataFrame(picks_h3fixed)

In [ ]:
tr = train_b.groupby(['h3_label', 'trees']).train_ic.mean()
rows = [{'kind': 'fixed', 'h3_label': lab, 'trees': k, **score(val_r, p), 'train_ic': tr.loc[(lab, k)]}
        for (lab, k), p in preds_b.items()]
rows.append({'kind': 'yearly', 'h3_label': '|'.join(picks_b.h3_label), 'trees': '|'.join(map(str, picks_b.trees)),
             **score(val_r, pred_yearly), 'train_ic': picks_b.train_ic.mean(),
             'persist_h3': persistence(picks_b.h3_label), 'random_h3': 1 / len(labels_b), 'trees_median': picks_b.trees.median()})
for lab in labels_b:
    pk = picks_h3fixed[picks_h3fixed.h3_label.eq(lab)].sort_values('fold')
    rows.append({'kind': 'h3_fixed_yearly_trees', 'h3_label': lab, 'trees': '|'.join(map(str, pk.trees)),
                 **score(val_r, pred_h3fixed[lab]), 'train_ic': pk.train_ic.mean(), 'trees_median': pk.trees.median()})
lgb = pd.DataFrame(rows)
lgb.to_csv(OUT_DIR / 'tuning_fixed_lgb.csv', index=False, encoding='utf-8-sig')
print('저장:', OUT_DIR / 'tuning_fixed_lgb.csv')
print('해마다 선택(H3·트리 수 모두) / H3만 고정(트리 수만 해마다)')
display(lgb[lgb.kind.ne('fixed')][['kind', 'h3_label', 'mean_ic', 'icir', 'ic_pos', 'train_ic', 'persist_h3', 'trees_median']]
        .assign(h3_label=lambda d: d.h3_label.where(d.kind.ne('yearly'), '해마다')))
fixed = lgb[lgb.kind.eq('fixed')]
print('고정 평균 IC (행 트리 수 100개마다, 열 H3)')
display(fixed[fixed.trees.astype(int) % 100 == 0].pivot(index='trees', columns='h3_label', values='mean_ic')[labels_b])

## 3. 비교

고정 1등과 해마다 선택의 차이, 그리고 1등 주변(바로 이웃한 H3·λ 또는 트리 ±10)의 평균. 차이가 0.002를 넘고 주변도 비슷하게 높을 때만 고정으로 바꾼다.

In [ ]:
def neighbors_lin(fixed, best):
    lams = list(LAMBDAS)
    i, j = labels.index(best.h3_label), lams.index(best.ridge_lambda)
    near = [(labels[a], lams[b]) for a in (i - 1, i, i + 1) for b in (j - 1, j, j + 1)
            if 0 <= a < len(labels) and 0 <= b < len(lams) and (a, b) != (i, j)]
    f = fixed.set_index(['h3_label', 'ridge_lambda']).mean_ic
    return float(np.mean([f.loc[k] for k in near]))


def neighbors_lgb(fixed, best):
    k = int(best.trees)
    f = fixed[fixed.h3_label.eq(best.h3_label)].set_index('trees').mean_ic
    return float(np.mean([f.loc[t] for t in (k - 10, k + 10) if t in f.index]))


out = []
for loss in LOSSES:
    part = lin[lin.loss.eq(loss)]
    fixed = part[part.kind.eq('fixed')].astype({'ridge_lambda': float})
    best = fixed.loc[fixed.mean_ic.idxmax()]
    yearly = part[part.kind.eq('yearly_6lambdas')].iloc[0]
    out.append({'모델': f'Ridge {loss}', '해마다 선택': yearly.mean_ic, '고정 1등': best.mean_ic,
                '고정 1등 값': f'{best.h3_label}, λ={best.ridge_lambda:g}', '1등 이웃 평균': neighbors_lin(fixed, best),
                '차이(고정−해마다)': best.mean_ic - yearly.mean_ic})
fixed = lgb[lgb.kind.eq('fixed')].astype({'trees': int})
best = fixed.loc[fixed.mean_ic.idxmax()]
yearly = lgb[lgb.kind.eq('yearly')].iloc[0]
out.append({'모델': 'LightGBM 순위 회귀 d3', '해마다 선택': yearly.mean_ic, '고정 1등': best.mean_ic,
            '고정 1등 값': f'{best.h3_label}, 트리 {best.trees}', '1등 이웃 평균': neighbors_lgb(fixed, best),
            '차이(고정−해마다)': best.mean_ic - yearly.mean_ic})
for r in lgb[lgb.kind.eq('h3_fixed_yearly_trees')].itertuples():
    out.append({'모델': f'LightGBM 순위 회귀 d3 · H3 {r.h3_label} 고정(트리 수만 해마다)', '해마다 선택': yearly.mean_ic,
                '고정 1등': r.mean_ic, '고정 1등 값': f'{r.h3_label}, 트리 해마다(중앙 {r.trees_median:g})',
                '1등 이웃 평균': np.nan, '차이(고정−해마다)': r.mean_ic - yearly.mean_ic})
summary = pd.DataFrame(out)
summary['판단'] = np.where(summary['차이(고정−해마다)'] > 0.002, '고정 검토', '해마다 선택 유지')
summary

## 4. 학습 시작 연도·내부 검증 기간 (약 40분, 1번 셀 다음 이 절만 돌려도 됨)

지수형을 빼서 학습을 2003년 대신 2001년부터 시작할 수 있다(2003년은 지수형 ① 때문이었음). 그리고 3번 결과에서 부스팅 트리 수를
직전 1년으로 고르면 흔들렸으므로(4~945개) 내부 검증을 3년으로 늘려 본다. 한 번에 하나씩 바꿔 원인을 나눈다.

| 구성 | 학습 | 내부 검증(조정값 고르기) | 예: 2015년 예측 |
| --- | --- | --- | --- |
| A 지금 | 2003 ~ Y−2 | Y−1 (1년) | 학습 2003\~2013, 검증 2014 |
| B 2001 시작 | 2001 ~ Y−2 | Y−1 (1년) | 학습 2001\~2013, 검증 2014 |
| C 2001 시작·검증 3년 | 2001 ~ Y−4 | Y−3 ~ Y−1 (3년) | 학습 2001\~2011, 검증 2012\~2014 |

- 바깥 검증(채점)은 그대로 2006\~2021 192개월. 고른 뒤 재학습 없음(C는 학습이 예측 연도와 3년 떨어짐).
- A→B = 자료 2년 추가 효과, B→C = 36개월로 안정적으로 고르는 이득 vs 최근 2년을 학습에서 빼는 손해.
- 대상: 선형 Ridge(z점수 입력, Huber①·②, H3 10 × λ 6 해마다 선택), LightGBM 순위 회귀 d3(H3 3 × 트리 1\~1000 해마다 선택).
  A는 기존 결과(0.0903, 0.0872, 0.0863)와 같아야 한다.
- 결과: `outputs/model_results/<AUTHOR>/tuning_window.csv`.

In [ ]:
WINDOWS = {'A_2003_inner1': (2003, 1), 'B_2001_inner1': (2001, 1), 'C_2001_inner3': (2001, 3)}   # (학습 시작 목표연도, 내부 검증 연수)


def window_masks(prep, valid_year, start, inner_years):
    """한 Fold의 행: 학습 start ~ Y−1−L, 내부 검증 Y−L ~ Y−1(학습 행 안), 바깥 검증 Y(수익률 있는 전 종목)."""
    d = prep['data']
    year, train, predict = d.year.to_numpy(), d.train.to_numpy(bool), d.predict.to_numpy(bool)
    hi = valid_year - 1 - inner_years
    return {'fit': train & (year >= start) & (year <= hi),
            'inner': train & (year >= valid_year - inner_years) & (year <= valid_year - 1),
            'out': predict & (year == valid_year), 'train_years': f'{start}~{hi}',
            'inner_years': f'{valid_year - inner_years}~{valid_year - 1}'}


if 'prep_z' not in globals():
    prep_z = mt.prepare_inputs(ROOT, config, input_transform='zscore')
    data_z, labels, decay_of = prep_z['data'], prep_z['sigma_labels'], mt._decay_of(prep_z)
    val_z = mt.validation_rows(prep_z, config)
if 'prep_r' not in globals():
    prep_r = mt.prepare_inputs(ROOT, config)
    data_r, labels_b, decay_b = prep_r['data'], mt.boosting_labels(prep_r), mt._decay_of(prep_r)
    val_r = mt.validation_rows(prep_r, config)
LAMBDAS6 = (0.0001, 0.001, 0.01, 0.1, 1.0, 10.0)          # 지금 Ridge 후보
WIN_LOSSES = ('huber_q999', 'huber_sigma135')
win_rows, win_picks = [], []
for wname, (start, L) in WINDOWS.items():
    # ---- 선형: Fold마다 H3 10 × λ 6 중 내부 검증 IC 최고(같으면 λ 큰 쪽 → H3 긴 기간)
    t0 = time.perf_counter()
    for loss in WIN_LOSSES:
        pred = np.full(len(val_z), np.nan)
        tics = []
        for f in FOLDS:
            m = window_masks(prep_z, f['valid_year'], start, L)
            ctx = mt.FoldContext(data_z, m['fit'], m['inner'], m['out'])
            y_fit, cands = ctx.y('monthly_zscore'), []
            for lab in labels:
                cols = ['x_size', 'x_bm', 'x_mom', 'x_sigma_' + lab]
                X = ctx.X(cols, 'fit')
                delta = ctx.q999('monthly_zscore') if loss == 'huber_q999' else ctx.sigma2(cols, 'monthly_zscore')['delta']
                warm = None
                for lam in LAMBDAS6:
                    fit = linear.fit_linear(X, y_fit, model='ridge', loss=loss, ridge_lambda=lam, delta=delta, start=warm)
                    warm = (fit['intercept'], fit['coef'])
                    cands.append((ctx.scorer.score(linear.predict_linear(fit, ctx.X(cols, 'inner')))[0], lam, decay_of[lab], lab, fit, cols))
            ic, lam, _, lab, fit, cols = max(cands, key=lambda c: (c[0], c[1], c[2]))
            pred[val_z.fold.eq(f['fold']).to_numpy()] = linear.predict_linear(fit, ctx.X(cols, 'out'))
            tics.append(ctx.train_scorer.score(linear.predict_linear(fit, ctx.X(cols, 'fit')))[0])
            win_picks.append({'window': wname, 'model': f'ridge {loss}', 'fold': f['fold'], 'train_years': m['train_years'],
                              'inner_years': m['inner_years'], 'n_train': ctx.n_train, 'h3_label': lab, 'ridge_lambda': lam, 'inner_ic': ic})
        pk = pd.DataFrame([p for p in win_picks if p['window'] == wname and p['model'] == f'ridge {loss}'])
        win_rows.append({'window': wname, 'model': f'ridge {loss}', **score(val_z, pred), 'train_ic': np.mean(tics),
                         'persist_h3': persistence(pk.h3_label), 'persist_lambda': persistence(pk.ridge_lambda)})
    print(f'{wname} 선형 {time.perf_counter() - t0:.0f}초')
    # ---- 부스팅: Fold마다 H3 3개 학습, (H3 × 트리 수) 중 내부 검증 IC 최고(choose_boosting)
    t0 = time.perf_counter()
    pred = np.full(len(val_r), np.nan)
    tics = []
    months_r = data_r.target_month.to_numpy()
    for f in FOLDS:
        m = window_masks(prep_r, f['valid_year'], start, L)
        ctx = mt.FoldContext(data_r, m['fit'], m['inner'], m['out'])
        fits = []
        for lab in labels_b:
            cols = ['x_size', 'x_bm', 'x_mom', 'x_sigma_' + lab]
            fit = boosting.fit_boosting('lgb_rankreg', ctx.X(cols, 'fit'), ctx.y('percentile'), ctx.m_fit, target='percentile',
                                        loss='mse', depth=3, learning_rate=0.01, inner_X=ctx.X(cols, 'inner'),
                                        inner_months=months_r[m['inner']], inner_score=lambda p: ctx.scorer.score(p)[0],
                                        train_score=lambda p: ctx.train_scorer.score(p)[0], n_threads=N_THREADS)
            fits.append({'h3_label': lab, 'decay': decay_b[lab], 'fitted': fit, 'curve': fit['inner_curve'], 'columns': cols})
        chosen, trees, _ = mt.choose_boosting(fits)
        pred[val_r.fold.eq(f['fold']).to_numpy()] = boosting.predict_boosting(chosen['fitted'], ctx.X(chosen['columns'], 'out'), trees)
        tics.append(chosen['fitted']['train_curve'][-(-trees // 10) - 1])
        win_picks.append({'window': wname, 'model': 'lgb_rankreg d3', 'fold': f['fold'], 'train_years': m['train_years'],
                          'inner_years': m['inner_years'], 'n_train': ctx.n_train, 'h3_label': chosen['h3_label'], 'trees': trees,
                          'inner_ic': chosen['inner_ic'] if 'inner_ic' in chosen else float(chosen['curve'][trees - 1])})
        print(f"  {wname} Fold {f['fold']:>2} ({f['valid_year']}) 학습 {m['train_years']} 검증 {m['inner_years']}: "
              f"{chosen['h3_label']}·트리 {trees}, {time.perf_counter() - t0:.0f}초")
    pk = pd.DataFrame([p for p in win_picks if p['window'] == wname and p['model'] == 'lgb_rankreg d3'])
    win_rows.append({'window': wname, 'model': 'lgb_rankreg d3', **score(val_r, pred), 'train_ic': np.mean(tics),
                     'persist_h3': persistence(pk.h3_label), 'trees_median': pk.trees.median(), 'trees_min': pk.trees.min(),
                     'trees_max': pk.trees.max()})
    print(f'{wname} 부스팅 {time.perf_counter() - t0:.0f}초')
win = pd.DataFrame(win_rows)
win_picks = pd.DataFrame(win_picks)
win.to_csv(OUT_DIR / 'tuning_window.csv', index=False, encoding='utf-8-sig')
win_picks.to_csv(OUT_DIR / 'tuning_window_picks.csv', index=False, encoding='utf-8-sig')
print('저장:', OUT_DIR / 'tuning_window.csv', OUT_DIR / 'tuning_window_picks.csv')

In [ ]:
print('모델 × 구성 평균 IC (A 지금, B 2001 시작, C 2001 시작·검증 3년)')
display(win.pivot(index='model', columns='window', values='mean_ic'))
print('ICIR')
display(win.pivot(index='model', columns='window', values='icir'))
print('부스팅이 고른 트리 수')
display(win[win.model.eq('lgb_rankreg d3')][['window', 'trees_median', 'trees_min', 'trees_max', 'persist_h3', 'train_ic']])
print('Fold별 고른 트리 수 (행 Fold, 열 구성)')
display(win_picks[win_picks.model.eq('lgb_rankreg d3')].pivot(index='fold', columns='window', values='trees'))